# 09 딥러닝 예측 모델 (Phase 1 — 3/3)

**40조건** × **LSTM, Autoformer, N-HiTS, iTransformer**

07·08 결과와 합쳐 **10모델 중 조건별 Best** 선정

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())


Torch device: cuda (NVIDIA GeForce RTX 5090, 32GB)
시계열: 165
학습 <= 201730 | 검증: [201731, 201732, 201733]
Best 선정 기준: MAPE


### ① DL 모델 실험

In [2]:
# Phase 1 — DL 4종: SBC 20조건 + ML 20조건
cache = DATA_PROCESSED / 'phase1_dl_results.parquet'
if cache.exists():
    results = pd.read_parquet(cache)
    summary, best = summarize_phase1(results)
    print('캐시 로드 |', len(results), 'rows')
else:
    sbc = run_phase1_all(df, feat_df, 'SBC_CLUSTER', 'SBC', models=DL_MODELS)
    ml = run_phase1_all(df, feat_df, 'ML_CLUSTER', 'ML', models=DL_MODELS)
    results = pd.concat([sbc, ml], ignore_index=True)
    summary, best = summarize_phase1(results)
    results.to_parquet(cache, index=False)
    summary.to_csv(DATA_PROCESSED / 'phase1_dl_results_summary.csv', index=False)
    best.to_csv(DATA_PROCESSED / 'phase1_dl_results_best.csv', index=False)
    print('완료 |', len(results), 'rows')

display(best.sort_values(['cluster_scheme', 'type', 'cluster']))
print('\n=== 알고리즘별 평균', RANK_METRIC.upper(), '===')
print(results.groupby(['cluster_scheme', 'model'])[RANK_METRIC].mean().unstack('cluster_scheme').round(2))


Phase1 SBC:   0%|          | 0/20 [00:00<?, ?it/s]

Seed set to 42


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


You are using a CUDA device ('NVIDIA GeForce RTX 5090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


C:\Users\kjh\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


완료 | 1320 rows


,cluster_scheme,type,cluster,best_model,mae_mean,rmse_mean,best_mape,mase_mean
2,ML,A,1,N-HiTS,2986.174130,4282.824867,111.024823,4.813861
4,ML,A,2,Autoformer,154171.223958,218906.931699,119.232515,4.013308
9,ML,A,3,LSTM,94671.578817,145336.642114,105.572540,4.120936
14,ML,A,4,N-HiTS,27234.144208,39752.376082,116.133387,4.141050
19,ML,B,1,iTransformer,2676.984952,3903.354697,106.935093,3.253106
20,ML,B,4,Autoformer,60239.058972,77532.866444,93.708042,3.535654
26,ML,C,1,N-HiTS,1831.610984,2692.992519,113.119537,3.209389
29,ML,C,2,LSTM,88947.812500,130693.191539,85.467947,3.834252
32,ML,C,4,Autoformer,33102.133112,43506.443090,67.168218,3.965262
38,ML,D,1,N-HiTS,2859.281156,4179.655933,162.771409,3.704384



=== 알고리즘별 평균 MAPE ===
cluster_scheme      ML     SBC
model                         
Autoformer      249.62  315.41
LSTM            130.91  135.61
N-HiTS          119.19  121.84
iTransformer    151.23  137.17


### ② Phase 1 통합 Best (10모델)

In [3]:
stat = pd.read_parquet(DATA_PROCESSED / 'phase1_stat_results.parquet')
ml = pd.read_parquet(DATA_PROCESSED / 'phase1_ml_results.parquet')
dl = pd.read_parquet(DATA_PROCESSED / 'phase1_dl_results.parquet')

phase1_all = pd.concat([stat, ml, dl], ignore_index=True)
phase1_summary, phase1_best = merge_phase1_best(stat, ml, dl)
phase1_all.to_parquet(DATA_PROCESSED / 'phase1_all_results.parquet', index=False)
phase1_summary.to_csv(DATA_PROCESSED / 'phase1_summary.csv', index=False)
phase1_best.to_csv(DATA_PROCESSED / 'phase1_best_per_condition.csv', index=False)
print('Phase1 통합 Best |', len(phase1_best), '조건')
display(phase1_best.sort_values(['cluster_scheme', 'type', 'cluster']))


Phase1 통합 Best | 35 조건


,cluster_scheme,type,cluster,best_model,mae_mean,rmse_mean,best_mape,mase_mean
8,ML,A,1,XGBoost,2462.732860,3361.175379,99.807025,4.524735
18,ML,A,2,XGBoost,142286.177083,192591.435250,105.969377,3.789674
28,ML,A,3,XGBoost,58266.037150,80798.303933,59.800775,2.536248
38,ML,A,4,XGBoost,22253.233989,32108.340274,96.838858,3.578794
48,ML,B,1,XGBoost,2051.172302,2990.261434,97.434245,2.898510
58,ML,B,4,XGBoost,40878.801764,58713.693148,75.042206,2.771052
60,ML,C,1,ARIMA,1893.020533,2704.057307,110.169941,3.273067
75,ML,C,2,RF,75887.663677,113124.954524,74.104331,3.271271
81,ML,C,4,Autoformer,33102.133112,43506.443090,67.168218,3.965262
98,ML,D,1,XGBoost,2553.850063,3761.929759,129.707018,4.190404
